# LET'S MAKE A PROPER RAG PIPELINE WITH LANGCHAIN  AND WITH A CHAT SCREEN

In [73]:
import os
import glob
from dotenv import load_dotenv
import gradio 
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import SystemMessage, HumanMessage, convert_to_messages

In [72]:
LLM_MODEL = "gpt-4o-mini"
load_dotenv(override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
knowledge_base = glob.glob("knowledge-base/*/")
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""
llm = ChatOpenAI(temperature=0, model_name=LLM_MODEL)

## STEP 1: LOAD THE KNOWLEDGE BASE

In [75]:
def fetch_documents():
    documents = []
    for folder in knowledge_base:
        document_type = os.path.basename(os.path.normpath(folder))
        loaded_documents = DirectoryLoader(folder, glob="**/*.md", show_progress=True, loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'}).load()
        for document in loaded_documents:
            document.metadata["document_type"] = document_type
            documents.append(document)
    return documents


In [76]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = text_splitter.split_documents(documents)
    return chunks

In [109]:
def store_chunks_to_vector_db(chunks):
    embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
    if os.path.exists('vector_db'):
        Chroma(collection_name='vector_db', embedding_function=embeddings).delete_collection()
    vectorstore = Chroma.from_documents(
        documents=chunks, embedding=embeddings, persist_directory="vector_db", collection_name="vector_db"
    )
    return vectorstore

fetched_documents = fetch_documents()
chunks = split_documents(fetched_documents)
vectorstore = store_chunks_to_vector_db(chunks)

100%|██████████| 12/12 [00:00<00:00, 4349.43it/s]


In [108]:
def get_relevant_context(query, top_k=5):
    context_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})
    relevant_context = context_retriever.invoke(query, k=top_k)
    return relevant_context


In [110]:
def generate_system_prompt(query):
    relevant_context = (get_relevant_context(query))
    print("relevant_context--->", relevant_context)
    system_prompt = SYSTEM_PROMPT.format(context=relevant_context)
    return system_prompt

In [111]:
def chat_with_context(query, history=[]):
    system_prompt = generate_system_prompt(query)
    print("-==-=>", system_prompt)
    messages = convert_to_messages([SystemMessage(content=system_prompt), HumanMessage(content=query)])
    messages.extend(history)
    print(messages)
    response = llm.invoke(messages)
    print(response)
    return response.content


In [ ]:
gradio.ChatInterface(fn=chat_with_context, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


relevant_context---> [Document(id='2cd6198d-a4b0-482f-8dc4-c7277c4ba68b', metadata={'source': 'knowledge-base/employees/Alex Thomson.md', 'document_type': 'employees'}, page_content="# HR Record\n\n# Alex Thomson\n\n## Summary\n- **Date of Birth:** March 15, 1995  \n- **Job Title:** Sales Development Representative (SDR)  \n- **Location:** Austin, Texas  \n\n## Insurellm Career Progression\n- **November 2022** - Joined Insurellm as a Sales Development Representative. Alex Thomson quickly adapted to the team, demonstrating exceptional communication and rapport-building skills.\n- **January 2023** - Promoted to Team Lead for special projects due to Alex's initiative in driving B2B customer outreach programs.  \n- **August 2023** - Developed a training module for new SDRs at Insurellm, enhancing onboarding processes based on feedback and strategies that Alex Thomson pioneered.  \n- **Current** - Continues to excel in the role, leading a small team of 5 SDRs while collaborating closely wit